# Baseline Training (+ MLFlow Instantiation)
Based on train_baseline.py, rewritten to be databricks-native.

install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow import MlflowClient
from databricks.feature_engineering import FeatureLookup
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
fe = FeatureEngineeringClient()

CATALOG = "mlo"
FT = f"{CATALOG}.features.weather_daily_v2"   

In [0]:
mlflow.set_registry_uri("databricks-uc")          # register to Unity Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.models")

In [0]:
lookups = [FeatureLookup(table_name=FT, lookup_key=["station"], timestamp_lookup_key="date")]
ts = fe.create_training_set(
        df=spark.table(f"{CATALOG}.features.weather_labels"),
        feature_lookups=lookups, label="Bad")
pdf = ts.load_df().toPandas().dropna().sort_values("date").reset_index(drop=True)

FEATURES = ["AWND", "TMAX", "TMIN", "PRCP_today", "PRCP_lag1", "PRCP_roll3", "TMAX_lag1"] #Additional Features from V1.

In [0]:
split_idx = int(len(pdf) * 0.8)
train_df, test_df = pdf.iloc[:split_idx], pdf.iloc[split_idx:]
print(f"\nTrain: {len(train_df)} rows ({train_df['date'].min().date()} -> {train_df['date'].max().date()})")
print(f"Test:  {len(test_df)} rows ({test_df['date'].min().date()} -> {test_df['date'].max().date()})")


In [0]:
Xtr, ytr = train_df[FEATURES], train_df["Bad"]
Xte, yte = test_df[FEATURES],  test_df["Bad"]

In [0]:
user = spark.sql("select current_user()").first()[0]
mlflow.set_experiment("/Shared/adsp32021_weather")   # shared
MODEL_NAME = f"{CATALOG}.models.weather_quality"

In [0]:
import os
from pathlib import Path

def git_commit():
    try:
        head = Path(os.getcwd(), ".git", "HEAD").read_text().strip()
        if head.startswith("ref:"):
            return Path(os.getcwd(), ".git", head.split(" ", 1)[1]).read_text().strip()
        return head
    except Exception:
        return "unknown"

GIT_COMMIT = git_commit()
FT_VERSION = spark.sql(f"DESCRIBE HISTORY {FT}").selectExpr("max(version) AS v").first()["v"]
print("commit:", GIT_COMMIT, "| v2 delta version:", FT_VERSION)

with mlflow.start_run(run_name="logreg_v2_tagged"):
    mlflow.set_tags({"feature_set_version": "v2", "model_type": "logreg",
                     "target": "Bad_t+1", "source": "baseline",
                     "git_commit": GIT_COMMIT})
    mlflow.log_params({"solver": "liblinear", "class_weight": "balanced",
                       "features": ",".join(FEATURES),
                       "n_features": len(FEATURES),
                       "feature_delta_version": FT_VERSION})
    model = Pipeline([("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                   solver="liblinear", random_state=42))])
    model.fit(Xtr, ytr)
    pred, proba = model.predict(Xte), model.predict_proba(Xte)[:, 1]
    mlflow.log_metric("f1",        f1_score(yte, pred, zero_division=0))
    mlflow.log_metric("accuracy",  accuracy_score(yte, pred))
    mlflow.log_metric("precision", precision_score(yte, pred, zero_division=0))
    mlflow.log_metric("recall",    recall_score(yte, pred, zero_division=0))
    if len(set(yte)) > 1:
        mlflow.log_metric("roc_auc", roc_auc_score(yte, proba))
    fe.log_model(model=model, artifact_path="model", flavor=mlflow.sklearn,
                 training_set=ts, registered_model_name=MODEL_NAME,
                 input_example=Xtr.head(3))   
    print("Logged and registered.")

In [0]:
c = MlflowClient()
latest = max(int(v.version) for v in c.search_model_versions(f"name='{MODEL_NAME}'"))
c.set_model_version_tag(MODEL_NAME, str(latest), "semver", "v0.2.0")     # ← bump
c.set_registered_model_alias(MODEL_NAME, "candidate", latest)            # ← NOT 'baseline'
print(f"Registered {MODEL_NAME} v{latest} (semver v0.2.0, alias 'candidate')")